# Lab 02 — Indexing & scale (HNSW), unrolled

Lab 01 searched by **brute force**: compare the query to *every* vector. Exact,
but O(N) per query — fine for 10 rows, hopeless for a million. An **HNSW** index
trades a little accuracy for a huge speedup. This notebook shows every step: the
index DDL, exact vs approximate search, and how we measure the trade-off.

> **Why random vectors here?** This lab is about the *index*, not embeddings.
> HNSW treats vectors as points — where they come from doesn't change how it
> behaves — so we use random unit vectors. That keeps the focus on the index and
> means we only need the **Postgres** tunnel (no embedder). In a real system the
> vectors come from an embedder (Lab 01); the index is identical.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
p = Path.cwd()
LABS = next(d for d in [p, *p.parents] if (d / "shared").exists())
if str(LABS) not in sys.path:
    sys.path.insert(0, str(LABS))
from shared.db import connect          # opens a pgvector-aware connection
N, DIM, K = 20000, 384, 10
print("N =", N, " dim =", DIM)

## 1. Make N random unit vectors
Normalising to length 1 means cosine distance (pgvector's `<=>`) behaves cleanly.

In [ ]:
rng = np.random.default_rng(0)
vecs = rng.standard_normal((N, DIM)).astype("float32")
vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
print(vecs.shape, "→ first vector norm =", round(float(np.linalg.norm(vecs[0])), 3))

## 2. Store them in pgvector
One row per vector. `COPY` streams them all in one shot (an `INSERT` per row would
be one network round-trip each — slow).

In [ ]:
with connect() as conn, conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS lab02_items")
    cur.execute(f"CREATE TABLE lab02_items (id int, embedding vector({DIM}))")
    with cur.copy("COPY lab02_items (id, embedding) FROM STDIN") as cp:
        for i, v in enumerate(vecs):
            cp.write_row((i, v))
    conn.commit()
print("stored", N, "vectors")

## 3. Exact search = the ground truth
`ORDER BY embedding <=> query LIMIT k` is k-nearest-neighbour. With **no index**
this is a full scan — exact, and what we'll measure HNSW against. We hold out 50
fresh random vectors as queries, and time each with `EXPLAIN (ANALYZE)`, which
reports **server-side** execution time (so the SSH tunnel doesn't pollute it).

In [ ]:
Q = 50
queries = rng.standard_normal((Q, DIM)).astype("float32")
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

def topk(cur, qv, k):
    cur.execute("SELECT id FROM lab02_items ORDER BY embedding <=> %s LIMIT %s", (qv, k))
    return {r[0] for r in cur.fetchall()}

def server_ms(cur, qv, k):
    # EXPLAIN ANALYZE reports the query's execution time ON THE SERVER (ms),
    # excluding the SSH-tunnel round-trip -- so the speed reflects the index, not the network.
    cur.execute("EXPLAIN (ANALYZE, TIMING ON, FORMAT JSON) "
                "SELECT id FROM lab02_items ORDER BY embedding <=> %s LIMIT %s", (qv, k))
    return cur.fetchone()[0][0]["Execution Time"]

with connect() as conn, conn.cursor() as cur:
    cur.execute("SET enable_indexscan = off")   # force the full scan -> ground truth
    cur.execute("SET enable_bitmapscan = off")
    exact = [topk(cur, q, K) for q in queries]
    flat_ms = float(np.mean([server_ms(cur, q, K) for q in queries]))
print(f"exact/flat search: {flat_ms:.2f} ms/query (server-side) over {N} vectors")

## 4. Build the HNSW index — the star of the lab
`vector_cosine_ops` builds it for the `<=>` (cosine) operator. `m` = edges per
node, `ef_construction` = how hard it searches while building (bigger = better
graph, slower build).

In [ ]:
with connect() as conn, conn.cursor() as cur:
    t = time.perf_counter()
    cur.execute("CREATE INDEX ON lab02_items USING hnsw (embedding vector_cosine_ops) "
                "WITH (m = 16, ef_construction = 64)")
    conn.commit()
print(f"HNSW index built in {time.perf_counter() - t:.1f}s")

## 5. Approximate search — and the knob that controls it
Now the same query uses the index. `hnsw.ef_search` is the query-time dial: higher
= search more of the graph = better recall, slower. **recall@k** = of the k the
index returned, how many were in the exact top-k.

In [ ]:
def eval_ef(cur, ef):
    cur.execute(f"SET hnsw.ef_search = {ef}")
    hits = sum(len(topk(cur, q, K) & exact[i]) for i, q in enumerate(queries))
    ms = float(np.mean([server_ms(cur, q, K) for q in queries]))
    return hits / (Q * K), ms

results = []
with connect() as conn, conn.cursor() as cur:
    cur.execute("SET enable_indexscan = on")     # let the planner use HNSW
    for ef in [10, 20, 40, 100, 200, 400]:
        recall, ms = eval_ef(cur, ef)
        results.append({"ef": ef, "recall": recall, "ms": ms})
        print(f"ef_search={ef:4d}   recall@{K}={recall:.3f}   {ms:.2f} ms/query")

## 6. The trade-off, plotted

In [ ]:
import matplotlib.pyplot as plt
ef = [r["ef"] for r in results]
rec = [r["recall"] for r in results]
lat = [r["ms"] for r in results]
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(ef, rec, "o-", color="tab:blue"); ax1.set_xlabel("hnsw.ef_search")
ax1.set_ylabel("recall@%d" % K, color="tab:blue"); ax1.set_ylim(0, 1.03)
ax1.axhline(1.0, ls=":", color="gray")
ax2 = ax1.twinx(); ax2.plot(ef, lat, "s--", color="tab:red")
ax2.set_ylabel("ms / query", color="tab:red")
ax1.set_title("HNSW: recall vs speed (higher ef_search → more recall, slower)")
fig.tight_layout(); plt.show()
print(f"flat/exact was {flat_ms:.1f} ms/query — HNSW is far faster at high recall.")

## What you saw
- **The index is just SQL:** `CREATE INDEX ... USING hnsw (embedding vector_cosine_ops)`.
- **Exact vs approximate:** disabling index scans gives ground truth; the index
  gives near-instant answers that may miss a few — `ef_search` tunes how many.
- **The curve:** recall climbs as `ef_search` rises, at the cost of latency. You
  pick the point on that curve your app needs.
- **Why recall is modest here:** random ~384-D vectors are a *worst case* for ANN —
  in high dimensions everything is nearly equidistant, so the graph struggles. Real
  embeddings **cluster**, so at the same `ef_search` you'd see much higher recall.
  The trade-off *shape* is the lesson, not the exact numbers.

> **How we timed it:** `EXPLAIN (ANALYZE)` reports each query's time *on the server*,
> so the speed reflects the **index**, not the network — that's why flat and HNSW
> differ clearly here. `bench.py` uses the same idea (batched) with real sentence
> embeddings instead of random vectors.